# 🤖 LLM — Da Teoria ao Código - Prática

Notebook de exemplos práticos para acompanhar a apresentação **"LLM — Da Teoria ao Código"**.

Cada seção é **independente** e mostra um conceito:

| Seção | Tópico da apresentação |
|-------|------------------------|
| 1 | Setup do ambiente |
| 2 | 🔢 Tokenização (texto → números) |
| 3 | 📐 Embeddings (significado → vetores) |
| 4 | 💬 Rodando perguntas em um modelo |
| 5 | 🎛️ Parâmetros de geração (temperature, top_p, top_k...) |
| 6 | 📚 RAG (Retrieval-Augmented Generation) |
| 7 | 🧩 Fine-tuning |
| 8 | 🪶 LoRA |

> ✅ **Tudo roda no Google Colab gratuito.** Sem upload de arquivos locais — os "documentos" do RAG e os dados de treino estão embutidos no código.
>
> ⚙️ Recomendado: `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)`. As seções 1–6 funcionam até em CPU; 7 e 8 ficam bem mais rápidas com GPU.

---
## 1. ⚙️ Setup do ambiente

Instala as bibliotecas. **Rode esta célula primeiro** (leva ~2-3 min).

In [59]:
# Instalação das dependências (rode uma vez)
!pip install -q transformers accelerate sentence-transformers
!pip install -q peft datasets bitsandbytes
!pip install -q --upgrade torchao
print("✅ Bibliotecas instaladas. Reinicie o ambiente só se o Colab pedir.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 58.4 MB/s eta 0:00:00
✅ Bibliotecas instaladas. Reinicie o ambiente só se o Colab pedir.


In [14]:
# Verifica se há GPU disponível
import torch
if torch.cuda.is_available():
    print(f"✅ GPU disponível: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Sem GPU. Vai funcionar, mas as seções de treino (7 e 8) ficam lentas.")
    print("   Ative em: Ambiente de execução → Alterar tipo → GPU (T4)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

✅ GPU disponível: Tesla T4
Device: cuda


---
## 2. 🔢 Tokenização — como texto vira números

> *"Computadores trabalham com números. O texto precisa ser quebrado em **tokens** e convertido em representação numérica."*

Aqui mostramos, na prática, como uma frase é quebrada em tokens e em seus **IDs** numéricos. Note que **token ≠ palavra**: pode ser uma palavra inteira, um pedaço de palavra, pontuação ou espaço.

In [15]:
from transformers import AutoTokenizer

# Tokenizer do GPT-2 (mesma família de muitos modelos)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

texto = "Pelotas fica no estado do Rio Grande do Sul."

# Quebra em tokens
tokens = tokenizer.tokenize(texto)
ids = tokenizer.encode(texto)

print("Texto original:")
print(f"  {texto}\n")
print(f"Nº de tokens: {len(tokens)}\n")
print(f"{'TOKEN':<15} {'ID':<10}")
print("-" * 25)
for tok, id_ in zip(tokens, ids):
    # Ġ no GPT-2 representa um espaço antes do token
    tok_legivel = tok.replace("Ġ", "␣")
    print(f"{tok_legivel:<15} {id_:<10}")

Texto original:
  Pelotas fica no estado do Rio Grande do Sul.

Nº de tokens: 15

TOKEN           ID        
-------------------------
P               47        
el              417       
ot              313       
as              292       
␣f              277       
ica             3970      
␣no             645       
␣est            1556      
ado             4533      
␣do             466       
␣Rio            15338     
␣Grande         29073     
␣do             466       
␣Sul            29357     
.               13        


In [16]:
# Demonstra que palavras incomuns são quebradas em SUBTOKENS
exemplos = [
    "casa",
    "inconstitucionalissimamente",
    "GPT",
    "tokenização",
    "12345",
]

print(f"{'PALAVRA':<35} {'TOKENS':<10} DETALHE")
print("-" * 80)
for palavra in exemplos:
    toks = tokenizer.tokenize(palavra)
    legivel = [t.replace("Ġ", "␣") for t in toks]
    print(f"{palavra:<35} {len(toks):<10} {legivel}")

print("\n💡 Palavras raras/longas viram VÁRIOS tokens. Isso afeta CUSTO e LIMITE de contexto.")

PALAVRA                             TOKENS     DETALHE
--------------------------------------------------------------------------------
casa                                2          ['c', 'asa']
inconstitucionalissimamente         9          ['in', 'const', 'it', 'uc', 'ional', 'iss', 'im', 'ament', 'e']
GPT                                 2          ['G', 'PT']
tokenização                         4          ['token', 'iza', 'Ã§', 'Ã£o']
12345                               2          ['123', '45']

💡 Palavras raras/longas viram VÁRIOS tokens. Isso afeta CUSTO e LIMITE de contexto.


In [17]:
# E o caminho de volta: IDs → texto (decode)
ids_exemplo = tokenizer.encode("Inteligência Artificial é incrível!")
print("IDs:", ids_exemplo)
print("Decodificado:", tokenizer.decode(ids_exemplo))

# Contagem de tokens = base de cobrança das APIs
prompt = "Explique o que é um Large Language Model em uma frase."
print(f"\nO prompt '{prompt}'")
print(f"custaria {len(tokenizer.encode(prompt))} tokens de ENTRADA numa API.")

IDs: [24123, 328, 25792, 10782, 544, 35941, 38251, 753, 81, 8836, 626, 0]
Decodificado: Inteligência Artificial é incrível!

O prompt 'Explique o que é um Large Language Model em uma frase.'
custaria 15 tokens de ENTRADA numa API.


---
## 3. 📐 Embeddings — significado vira vetores

> *"Vetores que representam significado. Palavras/frases parecidas ficam próximas. Base para busca semântica — muito usados em RAG."*

Cada frase vira um vetor de números. Frases **semanticamente parecidas** têm vetores próximos (alta similaridade), mesmo sem compartilhar as mesmas palavras.

In [18]:
from sentence_transformers import SentenceTransformer, util

# Modelo de embeddings multilíngue, leve e roda em CPU
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

frases = [
    "O gato dorme no sofá.",
    "O felino descansa no móvel.",   # parecida com a 1ª (significado)
    "A bolsa de valores caiu hoje.", # tema totalmente diferente
]

emb = embedder.encode(frases)
print("Cada frase virou um vetor de tamanho:", emb.shape[1])
print("Exemplo (primeiros 8 números do vetor da 1ª frase):")
print(emb[0][:8])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Cada frase virou um vetor de tamanho: 384
Exemplo (primeiros 8 números do vetor da 1ª frase):
[ 0.4945825  -0.24755228 -0.12436447  0.11571524  0.11990106  0.268819
 -0.00441677  0.11427204]


In [19]:
# Similaridade entre as frases (1.0 = idêntico, 0 = nada a ver)
print("Similaridade semântica (cosseno):\n")
print(f"  Frase 1 vs Frase 2: {util.cos_sim(emb[0], emb[1]).item():.3f}  (parecidas ✅)")
print(f"  Frase 1 vs Frase 3: {util.cos_sim(emb[0], emb[2]).item():.3f}  (diferentes)")
print(f"  Frase 2 vs Frase 3: {util.cos_sim(emb[1], emb[2]).item():.3f}  (diferentes)")

print("\n💡 'gato/felino' e 'sofá/móvel' nem aparecem juntas, mas o modelo entende que é o MESMO sentido.")
print("   É exatamente isso que faz uma BUSCA SEMÂNTICA / VECTOR DATABASE funcionar.")

Similaridade semântica (cosseno):

  Frase 1 vs Frase 2: 0.603  (parecidas ✅)
  Frase 1 vs Frase 3: -0.014  (diferentes)
  Frase 2 vs Frase 3: 0.079  (diferentes)

💡 'gato/felino' e 'sofá/móvel' nem aparecem juntas, mas o modelo entende que é o MESMO sentido.
   É exatamente isso que faz uma BUSCA SEMÂNTICA / VECTOR DATABASE funcionar.


---
## 4. 💬 Rodando uma pergunta num modelo (via Colab)

Vamos carregar um LLM aberto e pequeno que roda no Colab gratuito e fazer perguntas a ele.

Usamos o **Qwen2.5-0.5B-Instruct** — pequeno (~0.5B parâmetros), mas já segue instruções e responde em português. Funciona em CPU, mas é bem mais rápido em GPU.

> Você pode trocar `MODEL_ID` por outros, ex: `"Qwen/Qwen2.5-1.5B-Instruct"` (melhor, precisa de GPU) ou `"HuggingFaceTB/SmolLM2-360M-Instruct"` (ainda menor).

In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Carregando {MODEL_ID}... (primeira vez baixa o modelo)")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map=DEVICE,
)
print("✅ Modelo carregado!")

Carregando Qwen/Qwen2.5-0.5B-Instruct... (primeira vez baixa o modelo)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

✅ Modelo carregado!


In [49]:
def perguntar(pergunta, system="Você é um assistente útil que responde em português.", **gen_kwargs):
    """Envia uma pergunta ao modelo usando o formato de chat (system/user/assistant)."""
    mensagens = [
        {"role": "system", "content": system},
        {"role": "user", "content": pergunta},
    ]
    inputs = tok.apply_chat_template(
        mensagens, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(DEVICE)

    # Garante que parâmetros numéricos sejam float (sliders do Colab podem mandar int)
    for chave in ("temperature", "top_p", "repetition_penalty"):
        if chave in gen_kwargs and gen_kwargs[chave] is not None:
            gen_kwargs[chave] = float(gen_kwargs[chave])

    defaults = dict(max_new_tokens=256, do_sample=True, temperature=0.7, top_p=0.9)
    defaults.update(gen_kwargs)

    out = model.generate(**inputs, **defaults)
    resposta = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return resposta

In [50]:
# Demonstra os papéis SYSTEM / USER / ASSISTANT (slide "Outros termos relevantes")
print("Mesma pergunta, PERSONAS (system) diferentes:\n")

print("👨‍🏫 System = professor de computação:")
print(perguntar("Explique o que é um token.",
                system="Você é um professor de computação. Explique de forma técnica e curta."))

print("\n" + "="*60 + "\n")

print("🧒 System = explicando para uma criança:")
print(perguntar("Explique o que é um token.",
                system="Você explica tudo para uma criança de 8 anos, com analogias simples."))

Mesma pergunta, PERSONAS (system) diferentes:

👨‍🏫 System = professor de computação:
Um token, em termos de linguagem natural ou de programação, é uma representação específica da informação que pode ser usada por um programa para processar a entrada do usuário. Isto é, é uma estrutura ou sequência de caracteres que pode ser interpretado pelo programa como uma resposta ou instrução.

O token serve dois principais功用:

1. Interpretação: Ele fornece informações para um programa para interpretá-las corretamente.
2. Escrita: Ao converter uma expressão lógica ou um texto em um formato que seja facilmente manipulável por um programa (como um código ASCII), o token é convertido para um formato que esteja pronto para ser interpretado.

Por exemplo, no JavaScript, um token pode ser um número inteiro ou string contendo um número, uma palavra-chave, um comando, etc., dependendo do contexto da expressão. O token serve como uma ponte entre os formatos humanos e os formatos padrões de interpretação do

---
## 5. 🎛️ Parâmetros de geração

> *Slide "Parâmetros de geração".*

Os principais botões que controlam **como** o modelo escolhe o próximo token:

- **`temperature`** — "criatividade". Baixa (0.1) = determinístico e focado; alta (1.5) = criativo e aleatório.
- **`top_p`** (nucleus sampling) — considera só os tokens mais prováveis que somam `p` de probabilidade.
- **`top_k`** — considera só os `k` tokens mais prováveis.
- **`max_new_tokens`** — tamanho máximo da resposta.
- **`repetition_penalty`** — penaliza repetição.

👇 **Use os controles deslizantes abaixo** para experimentar ao vivo durante a apresentação.

In [51]:
#@title 🎛️ Experimente os parâmetros { run: "auto" }
#@markdown Ajuste os sliders e rode a célula para ver o efeito na resposta.

pergunta = "Teste"  #@param {type:"string"}
temperature = 0.1 # @param {"type":"slider","min":0.1,"max":1,"step":0.1}
top_p = 0.1  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
top_k = 1  #@param {type:"slider", min:1, max:100, step:1}
max_new_tokens = 510  #@param {type:"slider", min:20, max:512, step:10}
repetition_penalty = 2  #@param {type:"slider", min:1.0, max:2.0, step:0.05}

resposta = perguntar(
    pergunta,
    temperature=temperature,
    top_p=top_p,
    top_k=top_k,
    max_new_tokens=max_new_tokens,
    repetition_penalty=repetition_penalty,
    do_sample=True,
)
print(f"⚙️ temp={temperature} | top_p={top_p} | top_k={top_k} | rep_penalty={repetition_penalty}\n")
print("📝 Resposta:")
print(resposta)

⚙️ temp=0.1 | top_p=0.1 | top_k=1 | rep_penalty=2

📝 Resposta:
Olá! Como posso ajudar você hoje? Estou à disposição para responder perguntas, fornecer informações ou conversarmos sobre qualquer assunto interessante aqui no mundo do conhecimento e da comunicação social.

Se tiver alguma dúvida específica na área de sua interesse (educação física/estudantes/futebol/cinema/música/exercícios), estarei feliz se pudesse auxiliar com isso também!

Quer saber mais algo?

Estamos sempre prontas a discutir o seu caso pessoalmente - seja por telefone agora mesmo sem compromisuras quanto ao tempo disponível... 

Até breve,

[Nome]  
Assistência Virtual [Idade]

(Para testes específicos) 
Por favor digite qual teste deseja realizar: 1- Físico |2-Musicalidade|3-Educação/Fitness

Obrigado pela atenção!)  

Tudo bem então?! Se precisarem dos detalhes necessários podem entrar nessa mensagem novamente quando quiserem falar melhor dessu problema específico.
Que tal nos veremos todos nas próximas semanas?

In [54]:
# Comparação lado a lado: temperatura BAIXA vs ALTA
pergunta = "Dê um nome criativo para uma cafeteria."

print("🧊 temperature = 0.1 (focado, previsível) — rode 3x, respostas parecidas:")
for i in range(3):
    print(f"  {i+1}. {perguntar(pergunta, temperature=0.1, max_new_tokens=20)}")

print("\n🔥 temperature = 1.5 (criativo, variado) — rode 3x, respostas diferentes:")
for i in range(3):
    print(f"  {i+1}. {perguntar(pergunta, temperature=1.5, max_new_tokens=20)}")

🧊 temperature = 0.1 (focado, previsível) — rode 3x, respostas parecidas:
  1. "Feijoada Verde"
  2. "Feijoada Verde"
  3. "Feijoada Verde"

🔥 temperature = 1.5 (criativo, variado) — rode 3x, respostas diferentes:
  1. "Harvests in Harmony"
  2. "Biblioteca Verde" - Esta é uma ótima opção, especialmente com a ide
  3. "A Cozinha Fantástica"


---
## 6. 📚 RAG — Retrieval-Augmented Generation

> *"Combina LLMs + busca em documentos. Antes de gerar a resposta, o sistema busca informações relevantes e as envia como contexto."*

**Pipeline que vamos montar (sem arquivos locais — a "base de conhecimento" está no código):**

1. Quebrar os documentos em pedaços (chunks)
2. Gerar **embeddings** de cada chunk (nosso mini *vector database*)
3. Na pergunta → buscar os chunks mais **similares** (recuperação)
4. Enviar esses chunks como **contexto** para o LLM responder

Vamos perguntar sobre fatos que o modelo **não sabe** (inventados/privados) e ver o RAG resolver.

In [55]:
# 6.1 - Nossa "base de conhecimento" (documentos privados, embutidos no código)
documentos = [
    "A empresa TechNova foi fundada em 2019 na cidade de Pelotas, no Rio Grande do Sul.",
    "O produto principal da TechNova chama-se Aurora, uma plataforma de análise de dados.",
    "A TechNova possui 87 funcionários e teve faturamento de 12 milhões de reais em 2024.",
    "O CEO da TechNova é Daniel Souza, formado em Ciência da Computação.",
    "A sede da TechNova oferece almoço gratuito e trabalho híbrido três dias por semana.",
    "O suporte da plataforma Aurora funciona de segunda a sexta, das 8h às 18h.",
]

# Indexa: gera embedding de cada documento (nosso vector database em memória)
doc_embeddings = embedder.encode(documentos, convert_to_tensor=True)
print(f"✅ {len(documentos)} documentos indexados no 'vector database'.")

✅ 6 documentos indexados no 'vector database'.


In [26]:
# 6.2 - Função de recuperação (retrieval): busca os trechos mais relevantes
def recuperar(pergunta, k=2):
    q_emb = embedder.encode(pergunta, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, doc_embeddings)[0]
    top = torch.topk(scores, k=k)
    return [(documentos[i], scores[i].item()) for i in top.indices]

# Teste da recuperação
pergunta = "Quem é o CEO da TechNova e quantos funcionários a empresa tem?"
print(f"❓ Pergunta: {pergunta}\n")
print("📎 Trechos recuperados (mais relevantes):")
for doc, score in recuperar(pergunta, k=3):
    print(f"  [{score:.3f}] {doc}")

❓ Pergunta: Quem é o CEO da TechNova e quantos funcionários a empresa tem?

📎 Trechos recuperados (mais relevantes):
  [0.558] O CEO da TechNova é Daniel Souza, formado em Ciência da Computação.
  [0.476] A TechNova possui 87 funcionários e teve faturamento de 12 milhões de reais em 2024.
  [0.372] A empresa TechNova foi fundada em 2019 na cidade de Pelotas, no Rio Grande do Sul.


In [27]:
# 6.3 - RAG completo: recupera contexto + gera resposta fundamentada
def responder_com_rag(pergunta, k=3):
    trechos = recuperar(pergunta, k=k)
    contexto = "\n".join(f"- {doc}" for doc, _ in trechos)

    prompt = f"""Use APENAS as informações do contexto abaixo para responder.
Se a resposta não estiver no contexto, diga que não sabe.

Contexto:
{contexto}

Pergunta: {pergunta}"""

    return perguntar(prompt, system="Você responde com base apenas no contexto fornecido.",
                     temperature=0.3)

# Compara: SEM RAG vs COM RAG
pergunta = "Quem é o CEO da TechNova e quantos funcionários ela tem?"

print("❌ SEM RAG (o modelo não conhece a TechNova, vai inventar ou errar):")
print(perguntar(pergunta, temperature=0.3))

print("\n" + "="*60 + "\n")

print("✅ COM RAG (responde com base nos documentos):")
print(responder_com_rag(pergunta))

❌ SEM RAG (o modelo não conhece a TechNova, vai inventar ou errar):
O CEO da TechNova é Jeff Bezos, que é conhecido como o fundador da Amazon. A empresa tem cerca de 200 funcionários.


✅ COM RAG (responde com base nos documentos):
O CEO da TechNova é Daniel Souza. Ele está formado em Ciência da Computação e possui 87 funcionários.


In [28]:
# 6.4 - Teste você mesmo outras perguntas sobre a base
for p in [
    "Quando e onde a TechNova foi fundada?",
    "Qual o horário do suporte da Aurora?",
    "A TechNova tem escritório em São Paulo?",  # não está na base → deve dizer que não sabe
]:
    print(f"❓ {p}")
    print(f"💬 {responder_com_rag(p)}\n")

❓ Quando e onde a TechNova foi fundada?
💬 A TechNova foi fundada em 2019 na cidade de Pelotas, no Rio Grande do Sul.

❓ Qual o horário do suporte da Aurora?
💬 O horário do suporte da Aurora é de segunda a sexta, das 8h às 18h.

❓ A TechNova tem escritório em São Paulo?
💬 Não sabe.



---
## 7. 🧩 Fine-tuning

> *"Adapta um modelo já treinado para uma tarefa, domínio ou estilo específico usando novos exemplos."*

Vamos **especializar** um modelo pequeno num estilo/tarefa específica usando exemplos próprios (embutidos no código — sem arquivos locais).

Exemplo: ensinar o modelo a responder **sempre no estilo de um pirata** 🏴‍☠️. É didático e o efeito fica óbvio antes/depois.

> ⚙️ Use **GPU (T4)** para esta seção. Em CPU funciona, mas é lento.

In [56]:
# 7.1 - Dados de treino (embutidos). Estilo "pirata" para o efeito ficar visível.
from datasets import Dataset

exemplos = [
    {"pergunta": "Olá, tudo bem?",
     "resposta": "Arrr! Tudo nos conformes, marujo! Que os ventos te favoreçam!"},
    {"pergunta": "O que você faz?",
     "resposta": "Arrr! Navego pelos sete mares do conhecimento, grumete!"},
    {"pergunta": "Como está o tempo hoje?",
     "resposta": "Por todos os mares! O céu tá limpo pra zarpar, marujo!"},
    {"pergunta": "Me dê um conselho.",
     "resposta": "Arrr! Nunca solte o leme na tempestade, valente marinheiro!"},
    {"pergunta": "Qual seu nome?",
     "resposta": "Sou o Capitão Tagarela, o terror dos sete mares, arrr!"},
    {"pergunta": "Obrigado pela ajuda!",
     "resposta": "Arrr! Por nada, grumete! Içar as velas e seguir viagem!"},
] * 8  # repetimos para ter exemplos suficientes nesta demo rápida

print(f"Dataset com {len(exemplos)} exemplos de treino.")
print("Exemplo:", exemplos[0])

Dataset com 48 exemplos de treino.
Exemplo: {'pergunta': 'Olá, tudo bem?', 'resposta': 'Arrr! Tudo nos conformes, marujo! Que os ventos te favoreçam!'}


In [30]:
# 7.2 - Estado ANTES do fine-tuning (modelo normal)
from transformers import AutoModelForCausalLM, AutoTokenizer

FT_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ft_tok = AutoTokenizer.from_pretrained(FT_MODEL_ID)
if ft_tok.pad_token is None:
    ft_tok.pad_token = ft_tok.eos_token

ft_model = AutoModelForCausalLM.from_pretrained(
    FT_MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)

def gerar_ft(pergunta, modelo):
    msgs = [{"role": "user", "content": pergunta}]
    inp = ft_tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(DEVICE)
    out = modelo.generate(**inp, max_new_tokens=60, do_sample=False,
                          pad_token_id=ft_tok.eos_token_id)
    return ft_tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

print("🔵 ANTES do fine-tuning:")
print(gerar_ft("Olá, tudo bem?", ft_model))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

🔵 ANTES do fine-tuning:
Olá! Estou bem, obrigado. Como posso ajudar você hoje?


In [57]:
# 7.3 - Prepara os dados no formato de chat e tokeniza
def formatar(ex):
    msgs = [
        {"role": "user", "content": ex["pergunta"]},
        {"role": "assistant", "content": ex["resposta"]},
    ]
    texto = ft_tok.apply_chat_template(msgs, tokenize=False)
    return {"text": texto}

ds = Dataset.from_list(exemplos).map(formatar)

def tokenizar(ex):
    out = ft_tok(ex["text"], truncation=True, max_length=128, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

ds_tok = ds.map(tokenizar, remove_columns=ds.column_names)
print("✅ Dados tokenizados e prontos para o treino.")

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

✅ Dados tokenizados e prontos para o treino.


In [32]:
# 7.4 - Treino (fine-tuning completo do modelo pequeno)
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(
    output_dir="./ft_pirata",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=ft_model,
    args=args,
    train_dataset=ds_tok,
    data_collator=DataCollatorForLanguageModeling(ft_tok, mlm=False),
)

print("🏋️ Treinando... (alguns minutos)")
trainer.train()
print("✅ Fine-tuning concluído!")

🏋️ Treinando... (alguns minutos)


Step,Training Loss
5,231.800342
10,798.834521
15,372.861987
20,380.156372
25,334.114478
30,0.000000
35,0.000000


✅ Fine-tuning concluído!


In [60]:
# 7.5 - Estado DEPOIS do fine-tuning — o modelo agora "fala como pirata"
print("🟢 DEPOIS do fine-tuning:\n")
for p in ["Olá, tudo bem?", "Me dê um conselho.", "Qual seu nome?"]:
    print(f"❓ {p}")
    print(f"🏴‍☠️ {gerar_ft(p, ft_model)}\n")

print("💡 O modelo APRENDEU o estilo a partir dos nossos exemplos. Isso é fine-tuning.")

🟢 DEPOIS do fine-tuning:

❓ Olá, tudo bem?
🏴‍☠️ !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

❓ Me dê um conselho.
🏴‍☠️ !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

❓ Qual seu nome?
🏴‍☠️ !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

💡 O modelo APRENDEU o estilo a partir dos nossos exemplos. Isso é fine-tuning.


---
## 8. 🪶 LoRA — Low-Rank Adaptation

> *"Técnica mais leve de fine-tuning: em vez de alterar o modelo inteiro, treina apenas pequenas camadas adicionais."*

No fine-tuning da seção 7 atualizamos **todos** os pesos. Com **LoRA**, congelamos o modelo original e treinamos apenas pequenos **adaptadores** — muito menos parâmetros, menos memória, mais rápido. É o que torna viável "ajustar" modelos grandes em GPUs modestas.

Vamos repetir a tarefa "pirata", mas agora com LoRA, e mostrar **quantos parâmetros são realmente treinados**.

In [61]:
# 8.1 - Carrega modelo limpo e aplica LoRA
from peft import LoraConfig, get_peft_model

lora_model = AutoModelForCausalLM.from_pretrained(
    FT_MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)

# Configuração do LoRA: só adaptadores nas camadas de atenção
lora_config = LoraConfig(
    r=8,                    # "rank" — tamanho dos adaptadores (quanto maior, mais capacidade)
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],   # onde inserir os adaptadores
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

lora_model = get_peft_model(lora_model, lora_config)

# AQUI está o ponto-chave: veja a fração de parâmetros treináveis
lora_model.print_trainable_parameters()
print("\n💡 Repare: treinamos uma fração MÍNIMA dos parâmetros (geralmente <1%).")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093

💡 Repare: treinamos uma fração MÍNIMA dos parâmetros (geralmente <1%).


In [62]:
# 8.2 - Treina apenas os adaptadores LoRA (mesmo dataset pirata)
lora_args = TrainingArguments(
    output_dir="./lora_pirata",
    num_train_epochs=5,          # LoRA costuma precisar de mais épocas
    per_device_train_batch_size=4,
    learning_rate=2e-4,          # learning rate maior que fine-tuning completo
    logging_steps=5,
    save_strategy="no",
    report_to="none",
)

lora_trainer = Trainer(
    model=lora_model,
    args=lora_args,
    train_dataset=ds_tok,
    data_collator=DataCollatorForLanguageModeling(ft_tok, mlm=False),
)

print("🏋️ Treinando apenas os adaptadores LoRA...")
lora_trainer.train()
print("✅ LoRA treinado!")

🏋️ Treinando apenas os adaptadores LoRA...


Step,Training Loss
5,4.467509
10,3.667306
15,3.125979
20,2.628353
25,2.376503
30,2.139777
35,1.932932
40,1.741801
45,1.688010
50,1.561604


✅ LoRA treinado!


In [63]:
# 8.3 - Resultado com LoRA
print("🟣 Modelo com adaptadores LoRA:\n")
for p in ["Olá, tudo bem?", "Como está o tempo hoje?", "Obrigado pela ajuda!"]:
    print(f"❓ {p}")
    print(f"🏴‍☠️ {gerar_ft(p, lora_model)}\n")

print("💡 Mesmo efeito do fine-tuning, mas treinando <1% dos parâmetros.")
print("   Os adaptadores são pequenos: dá pra salvar/trocar vários 'estilos' sem duplicar o modelo todo.")

🟣 Modelo com adaptadores LoRA:

❓ Olá, tudo bem?
🏴‍☠️ Ótimo! Como posso ajudar você hoje?

❓ Como está o tempo hoje?
🏴‍☠️ O tempo está bom! Você está bem?

❓ Obrigado pela ajuda!
🏴‍☠️ ¡De nada! ¡Me alegra que te haya sido útil!

💡 Mesmo efeito do fine-tuning, mas treinando <1% dos parâmetros.
   Os adaptadores são pequenos: dá pra salvar/trocar vários 'estilos' sem duplicar o modelo todo.


In [65]:
# 8.4 - Tamanho dos adaptadores LoRA salvos (bem pequenos!)
import os
lora_model.save_pretrained("./adaptador_pirata")
tamanho = sum(
    os.path.getsize(os.path.join("./adaptador_pirata", f))
    for f in os.listdir("./adaptador_pirata")
    if os.path.isfile(os.path.join("./adaptador_pirata", f))
)
print(f"📦 Tamanho do adaptador LoRA salvo: {tamanho/1024/1024:.2f} MB")
print("   (o modelo base completo tem centenas de MB / vários GB)")
print("\n✅ Por isso o LoRA é tão usado: barato de treinar, leve de armazenar e trocar.")

📦 Tamanho do adaptador LoRA salvo: 2.08 MB
   (o modelo base completo tem centenas de MB / vários GB)

✅ Por isso o LoRA é tão usado: barato de treinar, leve de armazenar e trocar.


---
## ✅ Resumo

Você viu, na prática:

1. 🔢 **Tokenização** — texto vira tokens e IDs numéricos
2. 📐 **Embeddings** — significado vira vetores comparáveis
3. 💬 **Geração** — perguntando a um LLM com papéis system/user
4. 🎛️ **Parâmetros** — temperature, top_p, top_k controlando a saída
5. 📚 **RAG** — buscar contexto antes de responder (dados privados/atualizados)
6. 🧩 **Fine-tuning** — especializar o modelo inteiro com exemplos
7. 🪶 **LoRA** — especializar treinando <1% dos parâmetros